![Insper](https://github.com/danielscarvalho/Insper-DS-Dicas/blob/master/Insper-Logo.png?raw=true)

# Insper Pós-Graduação
## Programa Avançado em Data Science e Decisão [»](https://www.insper.edu.br/pos-graduacao/programas-avancados/data-science-e-decisao/)

# Aula 05 - Arquivos .SAV, gráficos com **altair**

Arquivo .SAV gerado com o "IBM SPSS STATISTICS"

## Pesquisa TIC Domicílios 2019

Realizada anualmente desde 2005, a pesquisa TIC Domicílios tem o objetivo de mapear o acesso às TIC nos domicílios urbanos e rurais do país e as suas formas de uso por indivíduos de 10 anos de idade ou mais.

A pesquisa TIC Domicílios conta com módulos fixos (coleta anual) e módulos rotativos (outras periodicidades). Os indicadores gerados pela pesquisa oferecem um cenário do acesso e do uso de TIC do Brasil, abordando diversos temas, tais como:

Acesso às TIC;
Uso do computador;
Uso da Internet;
Habilidades na Internet;
Uso do celular;
Governo eletrônico;
Comércio eletrônico;
Atividades culturais na Internet.

A última pesquisa disponível é a do ano de 2019, e todos os dados encontram-se disponíveis em https://cetic.br/pt/pesquisa/domicilios/microdados/

In [ ]:
#Talvez seja necessário no seu sistema...
#!conda install altair -y

In [ ]:
#Talvez seja necessário no seu sistema...
#!conda install pyreadstat -y

In [ ]:
import pandas as pd
import numpy as np
import pyreadstat
import altair as alt

O arquivo que vamos trabalhar possui a extensão .SAV, formato este proveniente do software SPSS (*Statistical Package for the Social Science*). Como o próprio nome sugere, é um software com pacote estatístico para construção, controle, inserção, obtenção de resultados e tomada de decisões baseadas em estatísticas. Entre muitos outros softwares, o SPSS caracteriza-se pela fácil utilização do usuário final, ou seja, como tudo que se deseja no programa pode ser feito através do sistema point and click (apontar e clicar), não é necessário que o usuário final seja programador, como acontece com muitos outros programas estatísticos. O arquivo .sav armazena todas as informações do banco de dados, como definição de variáveis e dados inseridos. 

Observe bem, ele armazena DADOS e DEFINIÇÃO DE VARIÁVEIS. E isso é ótimo para nós. Vamos ler este arquivo por meio da biblioteca pyreadstat.  Observe que nosso arquivo apresenta um encoding diferente ('latin1'). Você sabe explicar o que isso significa?


In [ ]:
#Linux, Mac, Unix
!head Bases/ticdom_2019_individuos_base_de_microdados_v1.0.SAV

Character Set

- ASCII
- ISO-8859-1, Lating1
- UTF-8 (Unicode)

In [ ]:
df, meta = pyreadstat.read_sav('Bases/ticdom_2019_individuos_base_de_microdados_v1.0.SAV', encoding='latin1')

In [ ]:
type(df)

In [ ]:
type(meta)

In [ ]:
df.head(5)

In [ ]:
df.shape

Observe que a leitura do arquivo .sav retorna dois objetos: um dataframe e um dicionário de metadados. É com esse dicionários que vamos escrever uma função denominada *convert_to_label* que irá retornar os valores categóricos para cada resposta previamente codificada. 

In [ ]:
meta.value_labels[
    meta.variable_to_label['RELIGIAO']][1.0]

In [ ]:
def calc(val):
    try:
        return 1/val
    except:
        return 0

In [ ]:
calc(0)

In [ ]:
1/"a"

In [ ]:
def convert_to_label(column, value, meta):
    try:
        return meta.value_labels[meta.variable_to_label[column]][value]
    except:
        return value

Vamos fazer uma cópia do dataframe.

In [ ]:
df_labels = df.copy()

In [ ]:
df_labels

In [ ]:
for c in list(df):
    df_labels[c] = df_labels[c].apply(lambda x: convert_to_label(c, x, meta))

In [ ]:
df_labels.head()

In [ ]:
pd.set_option('display.max_rows',1000) #define o numero maximo de linhas a serem exibidas no pandas

In [ ]:
df_labels.loc[0]

Python

- for
- lambda
- try
- list
- .apply
- def

## Medida de frequência ponderada

In [ ]:
# X é a variável que iremos contar frequencia de seus valores
# Y é a variável que possui o peso amostral

def weighted_frequency(x, y):
    a = pd.Series(df[[x,y]].groupby(x).sum()[y])/df[y].sum()
    b = a.index.map(meta.variable_value_labels[x])
    c = a.values
    df_temp = pd.DataFrame({'Labels': b, 'Frequency': c})
    return df_temp

In [ ]:
weighted_frequency('B1', 'PESO')

Vamos fazer uso da função *crosstab* da biblioteca Pandas para verificar a quantidade de respostas para o indicador B1 (já usou computador?), em função da Raça.

In [ ]:
df_labels['RACA']

In [ ]:
df_labels['B1']

In [ ]:
pd.crosstab(df_labels['RACA'], df_labels['B1'])

O que temos é a quantidade. E se nosso objetivo fosse obter a soma do peso amostral? E para cada possível valor da Raça, nós tivéssemos quanto isso representa em percentual?

In [ ]:
df_labels['RACA']

In [ ]:
df_labels['PESO']

In [ ]:
pd.crosstab(df_labels['RACA'], df_labels['B1'],
            df_labels['PESO'], aggfunc='sum', normalize='index')

In [ ]:
def gera_tabela_freq(df, dimensao, indice, peso):
    return pd.crosstab(df[dimensao], df[indice], df[peso], aggfunc='sum', normalize='index')

In [ ]:
df_labels['RELIGIAO']

In [ ]:
df_labels['B1']

In [ ]:
df_labels['PESO']

In [ ]:
gera_tabela_freq(df_labels, 'RELIGIAO', 'B1',  'PESO')

In [ ]:
gera_tabela_freq(df_labels, 'PEA_2', 'B1',  'PESO')

In [ ]:
[[a+b for a in range(10)] for b in range(10) if b%2 == 0]

In [ ]:
dimensoes = ['AREA', 'SEXO', 'COD_REGIAO_2', 'RACA', 'GRAU_INSTRUCAO',
             'FAIXA_ETARIA', 'RENDA_FAMILIAR',  'CLASSE_CB2015',  'RELIGIAO', 'PEA_2']

indicadores = [c for c in df_labels.columns if c not in dimensoes + ['QUEST', 'ID_DOMICILIO', 'ID_MORADOR',
                                                                     'IDADE', 'RENDA_PESSOAL', 'GRAU_INSTRUCAO_2', 'RENDA_FAMILIAR_2', 'PEA', 'ESTRATO', 'UPA',  'PEA',  'PESO', 'CLASSE_CB2008', ]]

In [ ]:
dimensao = 'SEXO'
indicador = 'B1'

In [ ]:
weighted_frequency(indicador, 'PESO')

In [ ]:
alt.Chart(weighted_frequency(indicador, 'PESO')).mark_bar().encode(
        x=alt.X('Labels', axis=alt.Axis(labelAngle=45, title='Respostas')),
        y=alt.Y('Frequency', axis=alt.Axis(title='Frequencia')),
        tooltip=['Labels', 'Frequency']
).interactive()

In [ ]:
gera_tabela_freq(df_labels, dimensao,  indicador, 'PESO').reset_index()

In [ ]:
dimensao = 'AREA'
tiny_data = pd.melt(gera_tabela_freq(df_labels, dimensao,  indicador, 'PESO').reset_index(), 
                    id_vars=dimensao, 
                    var_name='Respostas', 
                    value_name='Frequencia')

In [ ]:
tiny_data

In [ ]:
alt.Chart(tiny_data).mark_bar().encode(
        x=alt.X('Frequencia',  axis=alt.Axis(labelAngle=45, title='Frequencia')),
        column=dimensao,
        color='Respostas',
        y=alt.Y('Respostas', axis=alt.Axis(title='Respostas')),
).interactive()

In [ ]:
alt.Chart(tiny_data).mark_bar().encode(
        x=alt.X('Respostas',  axis=alt.Axis(labelAngle=45, title='Respostas')),
        column=dimensao,
        color='AREA',
        y=alt.Y('Frequencia', axis=alt.Axis(title='Frequencia')),
).interactive()

In [ ]:
def calc2(a=10, b=2):
    """Calc 2 faz calculo maluco!"""
    return a/2

In [ ]:
calc2()

In [ ]:
calc2(a=30)

In [ ]:
calc2(b=3)

In [ ]:
help(calc2)

In [ ]:
calc2(1,20)

In [ ]:
calc2(b=10,a=44)

### Referências

- https://github.com/Roche/pyreadstat
- https://altair-viz.github.io/